# Day 2 실습 — CoT·ReAct 단계적 추론과 정답률 비교

**목표**: 바로 답·CoT·Self-Consistency·ReAct로 단계적 추론을 직접 적용하고, 방법별 정답률을 측정·비교한다.
**구성**: Part 1 CoT 기초·오류 다루기 → Part 2 Self-Consistency·ReAct(+다른 도메인) → Part 3 정답률 비교 미니 프로젝트(+나만의 문제 세트)

> **참고:** 실습 전 가상환경 활성화, `.env`의 OpenAI API 키를 확인한다.

## 0. 환경 준비

이 셀은 노트북 전체에서 한 번만 실행한다. 문제는 두 개를 쓴다 — `problem`(연필 문제, 기본 난이도)과 `problem2`(기차 문제, 시간 단위 변환이 섞여 더 까다롭다).

In [9]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import re
from collections import Counter

# TODO: .env 파일의 환경변수를 불러오세요
load_dotenv()

#모델생성
llm = ChatOpenAI(model="gpt-4o-mini")
llm_low = ChatOpenAI(model="gpt-3.5-turbo")

#파서생성
parser = StrOutputParser()

In [7]:
problem = ("가게에 연필이 5타스 있다. (1타스는 12자루) "
           "오전에 40자루를 팔고, 오후에 2타스를 더 들여왔다. 지금 연필은 몇 자루인가?") # 44자루
problem2 = ("기차가 시속 80km로 2시간 30분을 달리다가, "
            "시속 60km로 1시간 12분을 더 달렸다. 총 이동 거리는 몇 km인가?") # 272 km

print("준비 완료:", llm.model_name)

준비 완료: gpt-4o-mini


## Part 1. CoT 기초

완성 코드를 직접 쳐서 바로 답과 CoT의 차이를 확인한다.

### 1-1. 바로 답

풀이 없이 답만 받는다. 까다로운 문제는 여기서 틀리기 쉽다.

In [14]:
# TODO: 풀이 없이 답만 숫자로 말하도록 지시하는 system 메시지를 작성하세요

direct = ChatPromptTemplate.from_messages([
    # ("system", "계산 결과를 숫자로 말하시오."),
    ("system", "계산 결과를 풀이 없이 답만 숫자로 말하시오."),
    ("human","{problem}")]
)

In [15]:
# 체인 실행
direct_chain = direct | llm | parser
direct_chain.invoke({"problem":problem})

'현재 연필은 44자루입니다.'

In [16]:
# 체인 실행
direct_chain = direct | llm_low | parser
direct_chain.invoke({"problem":problem})

'40자루를 팔았으므로 5타스 중 40자루가 줄었을 것입니다. 따라서 현재 연필은 5타스에서 40자루를 뺀 20자루가 남아 있습니다. 오후에 2타스를 더 들여왔으므로 이를 계산하면 20 + (2 * 12) = 44 자루입니다. 따라서 현재 연필은 44 자루가 있습니다.'

### 1-2. Zero-shot CoT

"단계별로 생각" 지시 한 줄로 풀이를 유도한다.

In [18]:
 # TODO: 단계별로 풀이한 뒤 마지막 줄에 '정답:'으로 답하도록 지시하는 system 메시지를 작성하세요
cot = ChatPromptTemplate.from_messages([
    ("system", "단계별로 풀이하고, 마지막 줄에 '정답:'으로 답하시오."),
    ("human","{problem}")]
)

# 체인 실행
zero_shot_chain = cot | llm | parser
print(zero_shot_chain.invoke({"problem":problem}))

1. 가게에 처음 있는 연필의 수를 계산합니다.
   - 연필은 5 타스가 있으므로, 5 타스 x 12 자루/타스 = 60 자루

2. 오전에 40 자루를 팝니다.
   - 현재 연필 수 = 60 자루 - 40 자루 = 20 자루

3. 오후에 2 타스를 더 들여옵니다.
   - 2 타스 x 12 자루/타스 = 24 자루
   - 현재 연필 수 = 20 자루 + 24 자루 = 44 자루

결과적으로 지금 가게에 있는 연필의 수는 44 자루입니다.

정답: 44


In [20]:
print((cot | llm_low | parser).invoke({"problem":problem}))

1. 5타스의 연필은 \(5 \times 12 = 60\)자루이다.
2. 오전에 40자루를 판 후 남은 연필은 \(60 - 40 = 20\)자루이다.
3. 오후에 2타스를 더 들인 후 연필의 총 개수는 \(20 + 2 \times 12 = 44\)자루이다.

따라서, 지금 가게에는 연필이 44자루 있다.

정답: 44


### 1-3. Few-shot CoT

풀이 예시를 먼저 보여줘 형식을 잡는다.

In [23]:
 # TODO: 위 질문의 풀이 예시를 ai 메시지로 작성하세요 (정답: 11)
 
few = ChatPromptTemplate.from_messages([
    ("system", "단계별로 풀이하고, 마지막 줄에 '정답:'으로 답하시오."),
    # 퓨샷추가
    ("human", "3 + 4 * 2 는 얼마야?"),
    ("ai", "곱셈 먼저 계산 : 4 * 2 = 8\n"
            "다음 덧셈 계산 : 3 + 8 = 11\n"
            "정답: 11"
     ),
    # 실제 질문
    ("human","{problem}")]
)

# 체인 실행
print((few | llm_low | parser).invoke({"problem":problem}))

판매 후 남은 연필 수 : 5타스 * 12자루/타스 - 40자루 = 60자루 - 40자루 = 20자루
들여온 연필 수: 2타스 * 12자루/타스 = 24자루
현재 연필 수는 20자루 + 24자루 = 44자루 입니다.

정답: 44


### 1-4. 정답 추출

'정답: <숫자>' 형식에서 숫자만 뽑는다 (뒤 Part의 채점 기초).

In [25]:
# TODO: cot 프롬프트로 problem을 풀게 한 뒤(체인 실행), '정답: <숫자>'에서 숫자만 뽑는 정규식으로 추출하세요
out = (few | llm | parser).invoke({"problem":problem})
out

'1. 가게에 있는 연필의 수를 구합니다.\n   - 5타스 = 5 * 12자루 = 60자루\n\n2. 오전에 판매한 연필 수를 뺍니다.\n   - 60자루 - 40자루 = 20자루\n\n3. 오후에 들어온 연필 수를 더합니다.\n   - 2타스 = 2 * 12자루 = 24자루\n   - 20자루 + 24자루 = 44자루\n\n따라서, 지금 연필은 44자루입니다.\n\n정답: 44'

In [28]:
nums = re.findall(r"정답:\s*([0-9]+)", out) # row string
nums[-1]

'44'

### 1-5. 바로 답 vs CoT 비교

기본 문제(`problem`)와 까다로운 문제(`problem2`) 모두에서 비교한다.

In [31]:
# TODO: problem·problem2 두 문제 모두에 대해 direct·cot 두 방법으로 각각 실행하고 결과를 출력하세요
print("--- problem (direct) ---")
print((direct | llm | parser).invoke({"problem":problem}))

print("--- problem (cot) ---")
print((cot | llm | parser).invoke({"problem":problem}))

--- problem (direct) ---
연필은 44자루입니다.
--- problem (cot) ---
1. 가게에 있는 연필의 수부터 계산합니다.  
   5타스는 12자루 * 5타스 = 60자루입니다.

2. 오전에 40자루를 팔았으니, 남은 연필의 수를 계산합니다.  
   60자루 - 40자루 = 20자루입니다.

3. 오후에 2타스를 더 들여왔으니, 이 연필을 자루로 계산합니다.  
   2타스는 12자루 * 2타스 = 24자루입니다.

4. 이제 현재 남은 연필 수에 오후에 추가된 연필을 더합니다.  
   20자루 + 24자루 = 44자루입니다.

따라서, 현재 남아있는 연필의 수는 44자루입니다.

정답: 44


In [34]:
print("--- problem2 (direct) ---")
print((direct | llm | parser).invoke({"problem":problem2}))

print("--- problem2 (cot) ---")
print((cot | llm | parser).invoke({"problem":problem2}))

--- problem2 (direct) ---
총 이동 거리는 220 km입니다.
--- problem2 (cot) ---
먼저, 각 구간의 이동 거리를 계산해 보겠습니다.

**1. 첫 번째 구간:**

- 속도: 시속 80 km
- 시간: 2시간 30분

2시간 30분을 시간으로 변환하면:
2시간 + 30분 = 2 + 0.5 = 2.5시간 

이동 거리 = 속도 × 시간 = 80 km/h × 2.5 h

이동 거리 = 200 km

**2. 두 번째 구간:**

- 속도: 시속 60 km
- 시간: 1시간 12분

1시간 12분을 시간으로 변환하면:
1시간 + 12분 = 1 + 12/60 = 1 + 0.2 = 1.2시간

이동 거리 = 속도 × 시간 = 60 km/h × 1.2 h

이동 거리 = 72 km

**3. 총 이동 거리:**

총 이동 거리 = 첫 번째 구간의 이동 거리 + 두 번째 구간의 이동 거리
총 이동 거리 = 200 km + 72 km = 272 km

따라서, 총 이동 거리는 **정답:** 272 km입니다.


### 1-6. 오류 다뤄보기 — 정답 추출이 실패하면?

지시를 느슨하게 주면 모델이 '정답:' 형식을 안 지킬 수 있다. 그러면 정규식이 아무것도 못 찾는다.

In [ ]:
# TODO: 일부러 형식 지시 없이 "풀이 과정을 알려줘" 정도로만 지시하는 system 메시지를 작성하세요
loose = None
# TODO: loose 프롬프트로 problem을 풀게 한 뒤, '정답: <숫자>' 정규식으로 추출해보세요 (형식이 없어 못 찾을 것이다)
loose_out = None

nums = None

print(loose_out)

### 1-7. 오류 다뤄보기 — 형식을 지켜도 매번 100%는 아니다

`cot`처럼 형식을 지시하면 실제로는 거의 항상 지켜진다. 하지만 이건 프롬프트 지시일 뿐 코드로 강제한 게 아니므로 100%를 보장하지는 않는다. 3번 실행해서 몇 번 형식을 지켰는지 직접 세어본다 — 3/3이 나와도 정상이다. 3/3은 "이번엔 안 깨졌다"는 뜻이지 "항상 보장된다"는 뜻은 아니다.

In [ ]:
checks = []
# TODO: 3번 반복 실행해 형식 준수 여부를 checks에 모으세요


print("형식 준수 여부:", checks)
print(f"형식 준수율: {sum(checks)}/{len(checks)}")

### 1-8. 오류 다뤄보기 — 모델은 맞았는데 내 정규식이 틀렸다면?

1-6·1-7은 **모델이 형식을 안 지킨** 경우였다. 이번엔 **모델은 정확히 형식을 지켰는데 내 코드(정규식)가 틀린** 경우를 본다. 원인이 다르므로 디버깅 방향도 다르다.

In [ ]:
# 형식을 지킨 정상 답변
out_ok = None
 
print("--- 답변 ---")
print(out_ok)

# TODO: 일부러 콜론(:)을 빼먹은 정규식을 써보세요
buggy_nums = None

print("버그 있는 정규식 결과:", buggy_nums[-1] if buggy_nums else None)

# TODO: 콜론을 포함한 올바른 정규식을 쓰세요
fixed_nums = None

print("올바른 정규식 결과:", fixed_nums[-1] if fixed_nums else None)

In [ ]:
# 정규식 예제

import re

text = "문의는 010-1234-5678 이나 01098765432, 010.9876.5432로 주세요."
# 번호 추출 패턴
pattern = r"010[-.]?\d{4}[-.]?\d{4}"

phone_numbers = re.findall(pattern, text)
print(phone_numbers)  # ['010-1234-5678', '01098765432']


['010-1234-5678', '01098765432', '010.9876.5432']


> **참고:** 프롬프트 지시만으로는 출력 형식이 100% 보장되지 않는다. 형식을 코드로 강제하는 방법은 Day03(Structured Output)에서 다룬다.

## Part 2. Self-Consistency·ReAct

CoT를 한 단계 더 밀어붙여 정답 안정성을 높이고, ReAct의 틀을 연습한다.

### 2-1. Self-Consistency — 여러 번 풀어 다수결

개별 시도는 흔들려도 다수결은 정답으로 모이는 경향을 본다.

In [41]:
# TODO: 단계별로 풀이한 뒤 마지막 줄에 "정답: <숫자>" 형식으로만 답하도록 지시하는 system 메시지를 작성하세요
cot_strict = None

cot_strict = ChatPromptTemplate.from_messages([
    ("system", "단계별로 풀이하고, 마지막 줄에 '정답:'으로 답하시오."),
    # 실제 질문
    ("human","{problem}")]
)

In [46]:
def extract_answer(text):
    # TODO: '정답: <숫자>'에서 숫자만 뽑는 정규식을 채우세요
    nums = re.findall(r"정답:\s*([0-9]+)", out)
    return nums[-1] if nums else None

In [50]:
# TODO: temperature=1.0인 llm을 만들고, cot_strict와 연결한 체인을 만드세요
llm_varied = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)
chain = cot_strict | llm_varied | parser

In [58]:
answers = [extract_answer(chain.invoke({"problem":problem})) for _ in range(5)]

In [ ]:

# TODO: 5번 실행해 답을 모으세요
print("각 시도:", answers)


각 시도: ['44', '44', '44', '44', '44']


In [62]:
# TODO: Counter로 가장 많이 나온 답을 구하세요
print("다수결:", Counter(answers).most_common(1))

다수결: [('44', 5)]


### 2-2. temperature 비교

0은 일관, 1은 다양 → 다수결이 의미를 가지려면 다양성이 필요하다.

In [66]:
# TODO: 비교할 temperature 두 값을 넣으세요 (예: 0.0, 1.0)
for temp in [0.0, 0.5, 1.0, 1.5]: # 2.0은 안쓴다.
    m = ChatOpenAI(model="gpt-4o-mini", temperature=temp)
    
    outs = [(cot_strict | m | parser).invoke({"problem":problem2}) for _ in range(3)]
    
    print(f"temperature={temp} 정답들:", [extract_answer(o) for o in outs])

temperature=0.0 정답들: ['44', '44', '44']
temperature=0.5 정답들: ['44', '44', '44']
temperature=1.0 정답들: ['44', '44', '44']
temperature=1.5 정답들: ['44', '44', '44']


### 2-3. ReAct 형식 — 생각·행동·관찰 틀 연습

도구 없이 형식만 연습한다. Observation은 모델이 스스로 지어낸다(틀릴 수 있음).

# Thought(생각) → Action(행동) → Observation(관찰) → Thought → ... → Answer(최종 답)

In [69]:
# TODO: Thought/Action/Observation/(반복 안내)/Answer 형식으로 추론하도록 지시하는 system 메시지를 작성하세요
react = ChatPromptTemplate.from_messages(
    [("system", "다음 형식으로 추론한다 : \n"
                "Thought: 지금 무엇을 생각하는지\n"
                "Action: 필요한 계산이나 행동\n"
                "Observation: 그 결과를 관찰\n"
                "필요하면 위의 3단계를 반복\n"
                "Answer: 최종 답"),
    ("human", "{problem}")
    ]
)

print((react | llm | parser).invoke({"problem":problem}))

Thought: 현재 연필의 개수를 계산하기 위해, 먼저 가게에 있는 연필의 초기 개수를 알아야 한다. 초기에는 5타스가 있고, 1타스는 12자루이므로 5타스는 총 몇 자루인지 계산해야 한다. 그리고 오전에 40자루를 팔고, 오후에 2타스를 추가로 들여왔다. 이 모든 과정을 통해 최종적으로 남아있는 연필의 수를 구할 것이다.

Action: 
1. 초기 연필 수 계산: 
   \( 5 \text{타스} \times 12 \text{자루/타스} = 60 \text{자루} \)

2. 오전에 팔린 연필 수에서 빼기: 
   \( 60 \text{자루} - 40 \text{자루} = 20 \text{자루} \)

3. 오후에 추가된 연필 수 계산: 
   \( 2 \text{타스} \times 12 \text{자루/타스} = 24 \text{자루} \)

4. 최종적으로 남아있는 연필 수 계산:
   \( 20 \text{자루} + 24 \text{자루} = 44 \text{자루} \)

Observation: 연필의 최종 개수는 44자루가 되었다.

Answer: 44자루


### 2-4. Self-Refine — 스스로 검토하고 고치기

Self-Consistency는 **여러 번 풀어 다수결**로 안정성을 높인다. Self-Refine은 **한 번 풀고, 스스로 검토해 고치는** 방식이다. 호출 횟수가 5배가 아니라 2배(1차 답변 + 검토)라 비용이 더 적게 든다.

In [70]:
draft = (cot | llm | parser).invoke({"problem": problem})
print("--- 1차 답변 ---")
print(draft)

--- 1차 답변 ---
1. 가게에 있는 연필의 초기 수를 계산합니다.
   - 1타스 = 12자루이므로, 5타스는:
     \( 5 \times 12 = 60 \) 자루입니다.

2. 오전에 팔린 연필 수를 빼줍니다.
   - 팔린 자루 수: 40자루
   - 남은 연필 수:
     \( 60 - 40 = 20 \) 자루입니다.

3. 오후에 추가된 연필의 수를 계산합니다.
   - 추가된 타스 수: 2타스
   - 2타스는:
     \( 2 \times 12 = 24 \) 자루입니다.

4. 오후에 추가된 연필을 더해줍니다.
   - 총 연필 수:
     \( 20 + 24 = 44 \) 자루입니다.

따라서, 가게에 현재 남아있는 연필은 44자루입니다.

정답: 44


In [74]:
# TODO: 1차 답변을 스스로 검토해 틀렸으면 고치라는 지시를 작성하는 system 메시지를 작성하세요
refine = ChatPromptTemplate.from_messages(
    [
        ("system", "아래는 문제와 1차 답변이다. 계산에 오류가 있는지 스스로 검토하시오."
                    "틀리면 고쳐서 다시 실행하고 '정답: <숫자>' 형식으로 답한다."
                    "맞으면 그대로 '정답: <숫자>' 형식으로 답한다."
        ),
        ("human", "문제:{problem}\n\n"
                    "1차 답변: {draft}"
         )
    ]
)

In [75]:
final = (refine | llm | parser).invoke({"problem": problem, "draft": draft})

print("--- 검토 후 최종 답변 ---")
print(final)

--- 검토 후 최종 답변 ---
1차 답변을 검토하겠습니다.

1. 가게에 있는 연필의 초기 수를 계산하는 과정은 정확합니다.
   - 5타스 → \( 5 \times 12 = 60 \) 자루

2. 오전에 판매된 연필 수를 빼주는 과정도 올바릅니다.
   - 남은 자루 수 → \( 60 - 40 = 20 \) 자루

3. 오후에 추가된 연필 수를 계산하는 부분도 맞습니다.
   - 2타스 → \( 2 \times 12 = 24 \) 자루

4. 오후에 추가된 연필을 더해주는 과정도 정확합니다.
   - 총 연필 수 → \( 20 + 24 = 44 \) 자루

모든 계산이 정확하므로, 최종 답변은 맞습니다.

정답: 44


## Part 2-확장. 다른 도메인에 적용하기 — 회의 시간 잡기 논리 퍼즐

산수 문제가 아니어도 CoT·Self-Consistency가 통하는지 확인한다. 이번 문제는 일정 조율이다.

### 2-5. 새 문제 정의

In [76]:
# TODO: 회의 시간을 정하는 논리 퍼즐 문제를 만드세요. 마지막에 '정답: HH:MM' 형식으로 답하라는 지시를 반드시 포함하세요
meeting_problem = ("팀 회의를 잡으려 한다. A는 9시~12시, B는 10시~13시, "
                   "C는 10시 30분~12시 30분에 회의가 가능하다. "
                   "1시간 회의를 가장 이르게 시작할 수 있는 시각은? "
                   "'정답: HH:MM' 형식으로 답하라.")

### 2-6. CoT로 풀기

In [87]:
# TODO: 일정 조율 문제를 단계별로 풀이한 뒤 '정답: HH:MM' 형식으로 답하도록 지시하는 system 메시지를 작성하세요
cot_meeting = ChatPromptTemplate.from_messages([
    ("system", "아래는 회의 일정의 조건입니다. 단계별로 풀이하고 마지막 줄에는 '정답: HH:MM' 형식으로 답합니다."),
    ("human","문제: {problem}\n\n")]
)


In [88]:
cot_meeting_result = (cot_meeting | llm | parser).invoke({"problem": meeting_problem})
print(cot_meeting_result)

회의를 잡기 위해 A, B, C의 가능한 시간을 살펴보겠습니다.

1. A의 가능 시간: 09:00 - 12:00
   - 09:00부터 시작해 1시간 회의를 할 경우, 09:00 - 10:00까지 가능합니다.
   - 10:00부터 시작해 1시간 회의를 할 경우, 10:00 - 11:00까지 가능합니다.
   - 11:00부터 시작해 1시간 회의를 할 경우, 11:00 - 12:00까지 가능합니다.

2. B의 가능 시간: 10:00 - 13:00
   - 10:00부터 시작할 경우, 10:00 - 11:00까지 가능합니다.
   - 11:00부터 시작할 경우, 11:00 - 12:00까지 가능합니다.
   - 12:00부터 시작할 경우, 12:00 - 13:00까지 가능합니다.

3. C의 가능 시간: 10:30 - 12:30
   - 10:30부터 시작할 경우, 10:30 - 11:30까지 가능합니다.
   - 11:30부터 시작할 경우, 11:30 - 12:30까지 가능합니다.

이제 모든 시간의 겹치는 부분을 살펴봐야 합니다. 회의는 A, B, C 세 사람이 모두 참석 가능해야 하므로, 각자의 가능한 시간 중 겹치는 시간을 찾아야 합니다.

A와 B의 겹치는 시간:
- 10:00 - 12:00

A와 C의 겹치는 시간:
- 10:30 - 12:00

B와 C의 겹치는 시간:
- 10:30 - 12:00

따라서, A, B, C가 모두 참석할 수 있는 시간은 다음과 같습니다:
- 10:30 - 12:00

1시간 회의를 가장 이르게 시작할 수 있는 시각은 10:30입니다.

정답: 10:30


### 2-7. Self-Consistency로 풀기

In [98]:
def extract_time(text):
    # TODO: '정답: HH:MM' 형식에서 시:분만 뽑는 정규식을 채우세요
    match = re.findall(r"정답:\s*(\d{1,2}:\d{2})", text)
    return match[-1] if match else None

extract_time("정답: 99:99")

'99:99'

In [99]:
# TODO: cot_meeting과 llm_varied를 연결한 체인을 만드세요
chain_meeting = cot_meeting | llm_varied | parser

# TODO: 5번 실행해 답을 모으세요
meeting_answers = [extract_time(chain_meeting.invoke({"problem":meeting_problem})) for _ in range(5)]

print("각 시도:", meeting_answers)
# TODO: Counter로 다수결을 구하세요
print("다수결:", Counter(meeting_answers).most_common(1))

각 시도: ['10:30', '10:30', '10:30', '10:30', '10:30']
다수결: [('10:30', 5)]


### 2-8. ReAct 형식을 새 도메인에도 적용

2-3에서 연필 문제에 썼던 ReAct 틀을 그대로 회의 문제에 적용한다.

In [94]:
# TODO: Thought/Action/Observation/(반복 안내)/Answer 형식으로 추론하도록 지시하는 system 메시지를 작성하세요
react_meeting = ChatPromptTemplate.from_messages(
    [("system", "다음 형식으로 추론한다 : \n"
                "Thought: 지금 무엇을 생각하는지\n"
                "Action: 필요한 계산이나 행동\n"
                "Observation: 그 결과를 관찰\n"
                "필요하면 위의 3단계를 반복\n"
                "Answer: 최종 답"),
    ("human", "{problem}")
    ]
)

print((react_meeting | llm | parser).invoke({"problem":meeting_problem}))

Thought: A, B, C가 모두 회의에 참여할 수 있는 시간을 파악해야 한다. 각 참여자의 가능한 시간대를 확인하고 그 교차점을 찾아야 한다.

Action: 
- A의 가능한 시간: 9:00~12:00
- B의 가능한 시간: 10:00~13:00
- C의 가능한 시간: 10:30~12:30

이 세 사람의 겹치는 시간대를 찾는다.  
A와 B의 교차 시간: 10:00~12:00  
A와 C의 교차 시간: 10:30~12:00  
B와 C의 교차 시간: 10:30~12:30  

이제 A, B, C 모두가 가능한 시간의 교차점을 찾는다.

Observation: 겹치는 시간대는 10:30~12:00이다. 그 중 가장 이른 시작 시각은 10:30이다.

Answer: 10:30


### 2-9. 표본 수(n)를 늘리면 다수결이 더 안정될까?

Self-Consistency의 표본 수 `n`을 3과 9로 바꿔가며 다수결이 얼마나 안정되는지 비교한다. `n`이 커질수록 비용도 그만큼 커진다.

In [100]:
# TODO: 비교할 표본 수 두 값을 넣으세요 (예: 3, 9)
for n in [3, 9]:
    # TODO: chain_meeting을 n번 실행해 답을 votes에 모으세요

    votes = [extract_time(chain_meeting.invoke({"problem":meeting_problem})) for _ in range(n)]

    print(f"n={n} 표본:", votes)
    # TODO: Counter로 다수결을 구하세요
    print(f"n={n} 다수결:", Counter(votes).most_common(1))

n=3 표본: ['10:30', '10:30', '10:30']
n=3 다수결: [('10:30', 3)]
n=9 표본: ['10:30', '10:30', '10:30', '10:30', '10:30', '10:30', '10:30', '10:30', '10:30']
n=9 다수결: [('10:30', 9)]


### 관찰 정리

- 산수 문제와 일정 조율 문제 모두에서, CoT·Self-Consistency·ReAct는 똑같이 통했는가?
- 도메인이 바뀌면서 정답 추출 정규식은 무엇이 달라졌는가?
- 표본 수(n)를 늘리면 항상 더 안정적일까? 비용 대비 이득은 어디까지가 적당해 보이는가?

## Part 3. 미니 프로젝트 — 추론 방법별 정답률 비교

문제 세트로 바로 답·CoT·Self-Consistency의 정답률을 직접 측정한다.

### 3-1. 문제 세트 준비

In [ ]:
problems = [
    {"q": "사무용품함에 볼펜이 23자루 있었다. 이번 주에 20자루를 각 팀에 나눠주고 6자루를 새로 구매했다. 지금 볼펜은 몇 자루인가?", "a": "9"},
    {"q": "화이트보드 마커가 5다스(1다스=12자루) 창고에 있었다. 이번 달에 40자루를 회의실에 배치하고 2다스를 추가로 구매했다. 지금 창고에 남은 마커는 몇 자루인가?", "a": "44"},
    {"q": "한 프로젝트팀에 총 32명이 배정되어 있다. 정규직 인원이 계약직 인원보다 6명 많다. 정규직은 몇 명인가?", "a": "19"},
    {"q": "탕비실에 커피믹스가 3개씩 4묶음 있었다. 그중 5개를 사용하고, 남은 것의 절반을 옆 팀에 나눠주면 몇 개를 나눠준 것인가?", "a": "3"},
    {"q": "시간당 60건을 처리하는 고객센터 상담원이 90분 동안 근무하면 총 몇 건을 처리하는가?", "a": "90"},
]

def extract(text):
    # TODO: 텍스트에서 마지막 숫자만 뽑는 정규식을 채우세요
    return None

### 3-2. 세 방법 구현

In [ ]:
# TODO: 설명 없이 정답만 출력하도록 지시하는 system 메시지를 작성하세요
direct_q = None

# TODO: 단계별 풀이 뒤 '정답: <숫자>'로 답하도록 지시하는 system 메시지를 작성하세요
cot_q = None


llm0 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm1 = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)

def solve_direct(q):
    # TODO: direct_q 프롬프트로 llm0을 호출해 답을 뽑고 extract()로 숫자만 추출해 반환하세요
    return None

def solve_cot(q):
    # TODO: cot_q 프롬프트로 llm0을 호출해 답을 뽑고 extract()로 숫자만 추출해 반환하세요
    return None

def solve_sc(q, n=5):
    # TODO: cot_q 프롬프트로 llm1을 n번 호출해 얻은 답들을 votes에 모으세요
    votes = None

    # TODO: votes에서 다수결로 가장 많이 나온 답을 반환하세요
    return None

### 3-3. 채점 실행

In [ ]:
# TODO: "바로 답"·"CoT"·"Self-Consistency" 세 이름을 각각 solve_direct·solve_cot·solve_sc 함수와 매핑한 딕셔너리를 만드세요
methods = None

for name, fn in methods.items():
    correct = 0
    # TODO: problems의 각 문제 p에 대해 fn(p["q"])가 p["a"]와 같으면 correct를 1씩 늘리세요

    # TODO: name·correct·전체 문제 수·정답률(%)을 출력하세요


### 3-4. 실제 호출 횟수 세어보기

"호출 횟수(비용) 1배·1배·5배"가 실제로 몇 번인지 문제 세트 크기에 맞춰 계산해본다.

In [ ]:
# TODO: 문제 세트(problems)의 길이를 n_problems에 저장하세요
n_problems = None

# TODO: 문제 수(n_problems)를 이용해 방법별 호출 수를 계산하세요
calls_direct = None
calls_cot = None
calls_sc = None

print(f"바로 답 호출 수: {calls_direct}")
print(f"CoT 호출 수: {calls_cot}")
print(f"Self-Consistency 호출 수: {calls_sc}")
print(f"세 방법 합계: {calls_direct + calls_cot + calls_sc}")

### 정답률 비교표 (직접 채우기)

위 실행 결과를 보고 채운다.

| 방법 | 정답 수 | 정답률 | 호출 횟수(비용) | 총평 |
| --- | --- | --- | --- | --- |
| 바로 답 | | | 1배 | |
| CoT | | | 1배 | |
| Self-Consistency | | | 5배 | |

**확인 질문**
- 정답률과 비용(호출 횟수)은 어떤 관계인가?
- 어떤 상황에서 Self-Consistency의 추가 비용이 정당화되는가?

## Part 3-확장. 실전 프로젝트 — 면접 코치 챗봇에 적용하기

Day01에서 만든 모의면접 코치를 오늘 배운 CoT·Self-Refine으로 업그레이드한다. 이번엔 정답이 하나로 정해지지 않으므로, 질문이 지원자 이력에 얼마나 잘 맞는지 직접 평가한다.

### 코치 챗봇 준비 (Day01 설정 재사용)

In [ ]:
candidate_info = """
[지원자 이력]
- 경력: 신입 (인턴 3개월)
- 사용 기술: Python, FastAPI, PostgreSQL
"""
interview_question = "위 지원자에게 물어볼 백엔드 개발자 면접 질문 5개를 만들어주세요."

### 방법 1 — 바로 (역할만, 기준선)

In [ ]:
# TODO: "너는 15년차 현직 백엔드 개발자 출신 모의면접 코치다"라는 system 메시지로 프롬프트를 작성하세요
baseline_coach = None

# TODO: baseline_coach와 llm을 연결해 interview_question으로 질문을 생성하고 출력하세요
baseline_questions = None
print(baseline_questions)

### 방법 2 — CoT: 근거를 대며 질문 생성

지원자 이력을 보고 왜 이 질문이 적절한지 이유를 먼저 생각한 뒤 질문을 만들게 한다.

In [ ]:
# TODO: 지원자 이력(candidate_info)을 보고 질문마다 이유를 먼저 생각한 뒤 번호로 제시하도록 지시하는 system 메시지를 작성하세요
cot_coach = None

# TODO: cot_coach와 llm을 연결해 candidate_info·interview_question으로 질문 초안을 생성하고 출력하세요
draft_questions = None
print(draft_questions)

### 방법 3 — Self-Refine: 난이도를 스스로 검토

CoT로 만든 질문 초안을 신입 지원자 수준에 맞는지 스스로 다시 검토하게 한다.

In [ ]:
# TODO: 질문 초안이 신입 수준(인턴 3개월)에 비해 너무 어려우면 더 쉽게 고치도록 지시하는 system 메시지를 작성하세요
refine_coach = None

# TODO: refine_coach와 llm을 연결해 candidate_info·draft_questions로 최종 질문을 생성하고 출력하세요
final_questions = None
print(final_questions)

### 정성적 비교표 (직접 채우기)

정답이 하나로 정해지지 않으므로 상/중/하로 직접 평가한다.

| 방법 | 지원자 수준 적합도 | 질문 다양성 | 총평 |
| --- | --- | --- | --- |
| 방법 1 (바로) | | | |
| 방법 2 (CoT) | | | |
| 방법 3 (Self-Refine) | | | |

**확인 질문**
- 세 방법 중 어떤 질문 목록이 신입 지원자에게 가장 적절했는가?
- 정답이 하나로 정해지지 않는 문제에서, 방법의 좋고 나쁨을 어떻게 판단할 것인가?

## 확인 문제

1. CoT와 ReAct는 각각 무엇을 하는가? 오늘은 어디까지 다루는가?
2. 단계적 사고가 정답률을 높이는 이유는 무엇인가?
3. Self-Consistency가 정답률을 높이는 원리는? `temperature`가 0이면 이 효과가 왜 줄어드는가?
4. 1-6·1-7·1-8에서 확인한 세 오류는 각각 원인이 무엇이었는가?
5. Part 2와 Part 2-확장에서, 도메인이 산수에서 일정 조율로 바뀌어도 변하지 않았던 것은 무엇인가?
6. Self-Consistency와 Self-Refine은 비용을 쓰는 방식이 어떻게 다른가?
7. Part 3-확장처럼 정답이 하나로 정해지지 않는 문제에서는 방법의 좋고 나쁨을 어떻게 판단해야 하는가?